In [6]:
import os
import sys

sys.path.append(os.path.join(os.getcwd(), "preprocessing", "label_mapping"))
from label_mapping import build_labeled_dataset

# mapping the label
df = build_labeled_dataset()


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26179 entries, 0 to 26178
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   image_path  26179 non-null  object
 1   label_it    26179 non-null  object
 2   label_en    26179 non-null  object
dtypes: object(3)
memory usage: 613.7+ KB


In [8]:
sys.path.append(os.path.join(os.getcwd(), "preprocessing", "data_loader"))
from data_loader import build_train_val_test_generators

# ImageDataGenerator resizes each image on load and rescales pixels to [0, 1]
# (normalization). 128x128 instead of 224 to train much faster on CPU, at a
# small accuracy cost. 70/15/15 train/val/test split.
train_generator, val_generator, test_generator = build_train_val_test_generators(
    df, project_root=os.getcwd(), image_size=(128, 128)
)

print(
    f"Train batches: {len(train_generator)}, "
    f"Val batches: {len(val_generator)}, "
    f"Test batches: {len(test_generator)}"
)
train_generator.class_indices

Found 18325 validated image filenames belonging to 10 classes.
Found 3927 validated image filenames belonging to 10 classes.
Found 3927 validated image filenames belonging to 10 classes.
Train batches: 573, Val batches: 123, Test batches: 123


{'butterfly': 0,
 'cat': 1,
 'chicken': 2,
 'cow': 3,
 'dog': 4,
 'elephant': 5,
 'horse': 6,
 'sheep': 7,
 'spider': 8,
 'squirrel': 9}

In [ ]:
sys.path.append(os.path.join(os.getcwd(), "model"))
from baseline_cnn import build_baseline_cnn

# input_shape and num_classes both read from the generator so the model always
# stays in sync with the data (image_shape is e.g. (128, 128, 3)).
model = build_baseline_cnn(
    input_shape=train_generator.image_shape,
    num_classes=len(train_generator.class_indices),
)
model.summary()

In [10]:
from tensorflow.keras.callbacks import EarlyStopping

# Stop once val_loss stops improving for 3 epochs, and restore the weights
# from the best epoch so we don't keep an overfitted final state.
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,
    callbacks=[early_stopping],
)

Epoch 1/30
573/573 ━━━━━━━━━━━━━━━━━━━━ 82s 141ms/step - accuracy: 0.2984 - loss: 1.9923 - val_accuracy: 0.3382 - val_loss: 1.9283
Epoch 2/30
573/573 ━━━━━━━━━━━━━━━━━━━━ 92s 160ms/step - accuracy: 0.4748 - loss: 1.5510 - val_accuracy: 0.5360 - val_loss: 1.3971
Epoch 3/30
573/573 ━━━━━━━━━━━━━━━━━━━━ 89s 156ms/step - accuracy: 0.5729 - loss: 1.2729 - val_accuracy: 0.5915 - val_loss: 1.2383
Epoch 4/30
573/573 ━━━━━━━━━━━━━━━━━━━━ 92s 161ms/step - accuracy: 0.6186 - loss: 1.1269 - val_accuracy: 0.6101 - val_loss: 1.2011
Epoch 5/30
573/573 ━━━━━━━━━━━━━━━━━━━━ 90s 157ms/step - accuracy: 0.6605 - loss: 1.0205 - val_accuracy: 0.6290 - val_loss: 1.1353
Epoch 6/30
573/573 ━━━━━━━━━━━━━━━━━━━━ 93s 163ms/step - accuracy: 0.6818 - loss: 0.9290 - val_accuracy: 0.6481 - val_loss: 1.0686
Epoch 7/30
573/573 ━━━━━━━━━━━━━━━━━━━━ 96s 167ms/step - accuracy: 0.7145 - loss: 0.8392 - val_accuracy: 0.6567 - val_loss: 1.0406
Epoch 8/30
573/573 ━━━━━━━━━━━━━━━━━━━━ 95s 166ms/step - accuracy: 0.7386 - loss: 0

In [11]:
import pandas as pd

# Per-epoch train vs val numbers. A widening train-minus-val accuracy gap
# (or val_loss rising while train_loss falls) is the overfitting signal.
history_df = pd.DataFrame(history.history)
history_df.index.name = "epoch"
history_df["acc_gap"] = history_df["accuracy"] - history_df["val_accuracy"]
history_df

,accuracy,loss,val_accuracy,val_loss,acc_gap
epoch,,,,,
0,0.298390,1.992344,0.338172,1.928313,-0.039781
1,0.474761,1.551018,0.536033,1.397100,-0.061271
2,0.572879,1.272913,0.591546,1.238342,-0.018667
3,0.618608,1.126919,0.610135,1.201091,0.008474
4,0.660464,1.020531,0.628979,1.135323,0.031485
5,0.681801,0.928959,0.648077,1.068647,0.033723
6,0.714488,0.839210,0.656735,1.040612,0.057753
7,0.738554,0.763340,0.669977,1.052974,0.068577
8,0.766603,0.690076,0.672778,1.057068,0.093825


In [12]:
# Honest score on data the model never saw during training or early stopping.
test_loss, test_accuracy = model.evaluate(test_generator)
print(f"Test accuracy: {test_accuracy:.4f} | Test loss: {test_loss:.4f}")

123/123 ━━━━━━━━━━━━━━━━━━━━ 22s 182ms/step - accuracy: 0.6718 - loss: 0.9919
Test accuracy: 0.6718 | Test loss: 0.9919


In [13]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# test_generator was built with shuffle=False, so model.predict iterates in the
# same order as test_generator.classes -> predictions line up with true labels.
class_names = list(test_generator.class_indices.keys())
y_true = test_generator.classes
y_pred = np.argmax(model.predict(test_generator), axis=1)

# Per-class precision/recall/f1 — shows which animals the model handles well.
print(classification_report(y_true, y_pred, target_names=class_names))

# Confusion matrix as a labeled table: rows = true class, cols = predicted.
# Big off-diagonal numbers = pairs the model mixes up (e.g. cat vs dog).
cm = confusion_matrix(y_true, y_pred)
pd.DataFrame(cm, index=class_names, columns=class_names)

123/123 ━━━━━━━━━━━━━━━━━━━━ 6s 51ms/step
              precision    recall  f1-score   support

   butterfly       0.79      0.81      0.80       317
         cat       0.50      0.41      0.45       250
     chicken       0.90      0.68      0.77       465
         cow       0.53      0.45      0.48       280
         dog       0.61      0.78      0.68       729
    elephant       0.76      0.52      0.62       217
       horse       0.65      0.63      0.64       394
       sheep       0.62      0.50      0.55       273
      spider       0.71      0.87      0.78       723
    squirrel       0.59      0.50      0.54       279

    accuracy                           0.67      3927
   macro avg       0.67      0.62      0.63      3927
weighted avg       0.68      0.67      0.67      3927



,butterfly,cat,chicken,cow,dog,elephant,horse,sheep,spider,squirrel
butterfly,256,4,4,1,11,0,4,0,34,3
cat,4,103,1,4,80,1,0,1,44,12
chicken,12,6,314,5,46,4,9,7,37,25
cow,1,4,3,126,49,5,51,23,12,6
dog,8,34,6,21,570,5,33,17,23,12
elephant,4,3,2,11,30,113,14,14,15,11
horse,2,7,3,43,54,10,248,12,12,3
sheep,0,5,6,25,40,7,14,136,30,10
spider,29,18,2,0,21,1,3,3,632,14
squirrel,8,22,6,4,35,2,4,7,51,140


In [14]:
sys.path.append(os.path.join(os.getcwd(), "evaluation", "model_metrics"))
from model_metrics import evaluate_model

# Model comparison sheet: one row per model, macro-averaged metrics.
# After training model #2 (transfer learning), append its row and re-run
# so both models sit side by side, then pick the best for the README.
model_results = []
model_results.append(evaluate_model(model, test_generator, "baseline_cnn"))

comparison_df = pd.DataFrame(model_results)
comparison_df.to_csv("model_comparison.csv", index=False)
comparison_df

123/123 ━━━━━━━━━━━━━━━━━━━━ 6s 45ms/step


,model,accuracy,precision,recall,f1
0,baseline_cnn,0.6718,0.6667,0.6151,0.6332
